# Uticaj gustine grafa, numeričke preciznosti i sistemskih optimizacija

Proširenje analize na nacionalnoj platformi za veštačku inteligenciju, kartica NVIDIA A100.

**Faza 1** meri faktor preklapanja i potrošnju memorije na četiri grafa različite gustine, sa tri geometrije uzorkovanja, šest veličina batch-a i tri numeričke preciznosti. Ne trenira se do konvergencije, jer je predmet analize memorija a ne tačnost.

**Faza 2** meri dobitak od sistemskih optimizacija: broja radnih procesa pri učitavanju, prevođenja modela kroz `torch.compile`, i formata TF32. Sve tri su preporuke iz 11. poglavlja knjige o efikasnom treningu, a kartica A100 podržava i bfloat16 i TF32, što Tesla T4 nije imala.

Napomene o postavci:

1. Graf ogbn-arxiv se pretvara u neusmeren, pa su sva četiri grafa neusmerena i uporediva po gustini.
2. Statistika stepena koristi ulazni stepen, jer se uzorkovanje širi po ulaznim potezima.
3. Pored naivne granice, računa se i granica koja uzima u obzir stvarne stepene čvorova, jer čvor sa tri suseda ne može dati petnaest.
4. Podešavanja učitavača se smanjuju sa rastom batch-a, da pripremljeni batch-evi ne prepune deljenu memoriju. To ne utiče na izmerene veličine, jer broj radnih procesa menja samo brzinu pripreme.
5. Rezultat svake konfiguracije se upisuje odmah, a već izmereno se preskače.

Priprema, jednom po instanci:

```
pip install torch_geometric ogb
pip install pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
```

In [1]:
!pip install torch_geometric ogb
!pip install pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.6.0+cu124.html

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html

[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
GRAFOVI = ['arxiv', 'products']#, 'yelp', 'reddit']
FAZA_1 = True
FAZA_2 = True
SEED = 42


In [3]:
import os, sys, time, json, gc, csv, random, shutil, logging
from pathlib import Path
from datetime import datetime

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.utils import to_undirected, degree
import torch_geometric.data

torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.storage.GlobalStorage,
])

device = torch.device('cuda')
REZULTATI = Path('rezultati_prosirenje')
REZULTATI.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.FileHandler(REZULTATI / 'izvrsavanje.log'), logging.StreamHandler(sys.stdout)],
    force=True)
log = logging.getLogger(__name__)

cc = torch.cuda.get_device_capability(0)
log.info(f'kartica {torch.cuda.get_device_name(0)}, compute capability {cc[0]}.{cc[1]}')
log.info(f'jezgara {os.cpu_count()}, deljena memorija {shutil.disk_usage("/dev/shm").total/1e9:.0f} GB')
log.info(f'nativni bfloat16: {cc[0] >= 8}, TF32: {cc[0] >= 8}')

def postavi_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

def upisi_red(fajl, red):
    p = REZULTATI / fajl
    novi = not p.exists()
    with open(p, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(red.keys()))
        if novi:
            w.writeheader()
        w.writerow(red)

def vec_uradjeno(fajl, kolone):
    p = REZULTATI / fajl
    if not p.exists():
        return set()
    with open(p) as f:
        return {tuple(str(r[k]) for k in kolone) for r in csv.DictReader(f)}

/home/nikola.vukotic.akademijanis/.local/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
/home/nikola.vukotic.akademijanis/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-07-29 23:04:21,904 [INFO] kartica NVIDIA A100-SXM4-40GB, compute capability 8.0
2026-07-29 23:04:21,905 [INFO] jezgara 256, deljena memorija 34 GB
2026-07-29 23:04:21,905 [INFO] nativni bfloat16: True, TF32: True


## Učitavanje grafova

Grafovi se učitavaju jedan po jedan i oslobađaju pre sledećeg. Arxiv se pretvara u neusmeren, ostali su već neusmereni.

In [4]:
def ucitaj_graf(ime):
    """Vraca data, indekse trening cvorova, broj klasa i oznaku da li je multi-label."""
    if ime in ('arxiv', 'products'):
        from ogb.nodeproppred import PygNodePropPredDataset
        ds = PygNodePropPredDataset(name=f'ogbn-{ime}', root='podaci')
        d = ds[0]
        d.y = d.y.squeeze()
        if ime == 'arxiv':
            d.edge_index = to_undirected(d.edge_index)
        idx = ds.get_idx_split()['train']
        klasa, multi = ds.num_classes, False
    elif ime == 'reddit':
        from torch_geometric.datasets import Reddit
        ds = Reddit(root='podaci/Reddit')
        d = ds[0]
        idx = d.train_mask.nonzero(as_tuple=True)[0]
        klasa, multi = ds.num_classes, False
    elif ime == 'yelp':
        from torch_geometric.datasets import Yelp
        ds = Yelp(root='podaci/Yelp')
        d = ds[0]
        idx = d.train_mask.nonzero(as_tuple=True)[0]
        klasa, multi = d.y.size(1), True
    else:
        raise ValueError(ime)
    d.train_mask = torch.zeros(d.num_nodes, dtype=torch.bool)
    d.train_mask[idx] = True
    return d, idx, klasa, multi


def ucitavanje_za(B):
    """Manje pripremanja unapred za velike batch-eve, da deljena memorija ne prekipi."""
    if B <= 512:
        return dict(num_workers=8, persistent_workers=True, prefetch_factor=4, pin_memory=True)
    if B <= 2048:
        return dict(num_workers=4, persistent_workers=True, prefetch_factor=2, pin_memory=True)
    return dict(num_workers=2, persistent_workers=True, prefetch_factor=1, pin_memory=True)


def oslobodi(*imena):
    for n in imena:
        if n in globals():
            del globals()[n]
    gc.collect()
    torch.cuda.empty_cache()

## Model i merenje

In [5]:
SKRIVENI, LR, N_UZORAKA = 256, 0.01, 20
TIPOVI = {'fp32': None, 'fp16': torch.float16, 'bf16': torch.bfloat16}


class GraphSAGE(torch.nn.Module):
    def __init__(self, ulaz, skriveni, izlaz, slojeva, dropout=0.5):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.slojeva, self.dropout = slojeva, dropout
        self.convs.append(SAGEConv(ulaz, skriveni))
        self.bns.append(torch.nn.BatchNorm1d(skriveni))
        for _ in range(slojeva - 2):
            self.convs.append(SAGEConv(skriveni, skriveni))
            self.bns.append(torch.nn.BatchNorm1d(skriveni))
        self.convs.append(SAGEConv(skriveni, izlaz))

    def forward(self, x, edge_index):
        for i in range(self.slojeva - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.convs[-1](x, edge_index)


def statistika_stepena(d, S):
    """Naivna granica po ciljanom cvoru i granica koja uzima u obzir stvarne stepene."""
    st = degree(d.edge_index[1], num_nodes=d.num_nodes).float()
    zbir = st.sum()
    naivna, p = 1.0, 1.0
    for s in S:
        p *= s
        naivna += p
    ef = [torch.clamp(st, max=s).mean().item() if i == 0
          else (st * torch.clamp(st, max=s)).sum().item() / zbir.item()
          for i, s in enumerate(S)]
    sa_st, p = 1.0, 1.0
    for e in ef:
        p *= e
        sa_st += p
    return naivna, sa_st, ef, st


def izmeri(d, idx, klasa, multi, S, B, preciznost, num_workers=None,
           koristi_compile=False, tf32=False, n_uzoraka=None, zagrevanje=5):
    """Meri broj cvorova i potega po batch-u, memoriju i vreme po batch-u."""
    n_uzoraka = n_uzoraka or N_UZORAKA
    torch.backends.cuda.matmul.allow_tf32 = tf32
    torch.backends.cudnn.allow_tf32 = tf32
    postavi_seed()
    kw = ucitavanje_za(B)
    if num_workers is not None:
        kw = dict(kw, num_workers=num_workers)
        if num_workers == 0:
            kw.pop('persistent_workers', None)
            kw.pop('prefetch_factor', None)
    loader = NeighborLoader(d, num_neighbors=list(S), input_nodes=idx,
                            batch_size=B, shuffle=True, **kw)
    model = GraphSAGE(d.num_features, SKRIVENI, klasa, len(S)).to(device)
    if koristi_compile:
        model = torch.compile(model, dynamic=True)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    tip = TIPOVI[preciznost]
    scaler = torch.amp.GradScaler('cuda', enabled=(preciznost == 'fp16'))
    model.train()
    torch.cuda.reset_peak_memory_stats()

    cvorova, potega, oom, t0, mereno = [], [], False, None, 0
    for i, batch in enumerate(loader):
        if i >= n_uzoraka + zagrevanje:
            break
        if i == zagrevanje:
            torch.cuda.synchronize()
            t0 = time.time()
        if i >= zagrevanje:
            cvorova.append(batch.num_nodes)
            potega.append(batch.num_edges)
        try:
            batch = batch.to(device, non_blocking=True)
            opt.zero_grad()
            with torch.autocast('cuda', dtype=tip or torch.float32, enabled=(tip is not None)):
                out = model(batch.x, batch.edge_index)[:batch.batch_size]
                y = batch.y[:batch.batch_size]
                gubitak = (F.binary_cross_entropy_with_logits(out, y.float()) if multi
                           else F.cross_entropy(out, y))
            scaler.scale(gubitak).backward()
            scaler.step(opt)
            scaler.update()
            if i >= zagrevanje:
                mereno += 1
        except torch.cuda.OutOfMemoryError:
            oom = True
            torch.cuda.empty_cache()
            break
    if t0 is not None:
        torch.cuda.synchronize()
    trajanje = (time.time() - t0) if (t0 is not None and mereno) else float('nan')
    vrh = torch.cuda.max_memory_allocated() / 1e6

    del model, opt, loader
    gc.collect()
    torch.cuda.empty_cache()
    return {'cvorova': float(np.mean(cvorova)) if cvorova else 0,
            'potega': float(np.mean(potega)) if potega else 0,
            'memorija_mb': vrh, 'oom': oom,
            'ms_po_batchu': (trajanje / mereno * 1000) if mereno else float('nan'),
            'uzoraka': len(cvorova)}

## Faza 1: gustina grafa i numerička preciznost

Četiri grafa, tri geometrije uzorkovanja, šest veličina batch-a, tri preciznosti.

In [6]:
FANOUTS = {'mali': [5, 5, 5], 'standardni': [15, 10, 5], 'veliki': [25, 15, 10]}
BATCHEVI = [32, 128, 512, 1024, 2048, 4096]
PRECIZNOSTI = ['fp32', 'fp16', 'bf16']

if FAZA_1:
    uradjeno = vec_uradjeno('faza1.csv', ['graf', 'fanout', 'batch', 'preciznost'])
    for ime in GRAFOVI:
        log.info('=' * 70)
        log.info(f'FAZA 1 | graf {ime}')
        d, idx, klasa, multi = ucitaj_graf(ime)
        ulazni = degree(d.edge_index[1], num_nodes=d.num_nodes).float()
        log.info(f'  {d.num_nodes:,} cvorova, {d.num_edges:,} potega, {d.num_features} atributa, '
                 f'{klasa} klasa, multi-label {multi}')
        log.info(f'  ulazni stepen: prosek {ulazni.mean():.2f}, medijana {ulazni.median():.0f}, '
                 f'najveci {ulazni.max():.0f}')

        for ime_f, S in FANOUTS.items():
            naivna, sa_st, ef, st_deg = statistika_stepena(d, S)
            upisi_red('faza1_stepen.csv', {
                'graf': ime, 'fanout': ime_f, 'S': str(S),
                'cvorova': d.num_nodes, 'potega': d.num_edges, 'atributa': d.num_features,
                'prosecan_ulazni_stepen': ulazni.mean().item(),
                'medijana_stepena': ulazni.median().item(), 'najveci_stepen': ulazni.max().item(),
                'granica_naivna': naivna, 'granica_sa_stepenom': sa_st,
                'efektivni_fanout': str([round(x, 2) for x in ef]),
                'udeo_ispod_S1_pct': (st_deg < S[0]).float().mean().item() * 100})
            log.info(f'  {ime_f} S={S}: granica po cvoru naivna {naivna:.0f}, '
                     f'sa stepenom {sa_st:.0f}, efektivni fanout {[round(x,2) for x in ef]}')

            for B in BATCHEVI:
                for prec in PRECIZNOSTI:
                    if (ime, ime_f, str(B), prec) in uradjeno:
                        continue
                    r = izmeri(d, idx, klasa, multi, S, B, prec)
                    nm, ns = B * naivna, B * sa_st
                    n = r['cvorova']
                    upisi_red('faza1.csv', {
                        'graf': ime, 'fanout': ime_f, 'S': str(S), 'batch': B,
                        'preciznost': prec, 'n_max_naivno': nm, 'n_max_stepen': ns,
                        'n_actual': '' if r['oom'] else n,
                        'eta_naivno_pct': '' if r['oom'] else (1 - n / nm) * 100,
                        'eta_stepen_pct': '' if r['oom'] else (1 - n / ns) * 100,
                        'potega': '' if r['oom'] else r['potega'],
                        'memorija_mb': '' if r['oom'] else r['memorija_mb'],
                        'ms_po_batchu': '' if r['oom'] else r['ms_po_batchu'],
                        'oom': r['oom'], 'uzoraka': r['uzoraka']})
                    if r['oom']:
                        log.warning(f'    B={B:5d} {prec}: nedostatak memorije')
                    else:
                        log.info(f'    B={B:5d} {prec}: N={n:9.0f} '
                                 f'eta_naiv={(1-n/nm)*100:5.1f}% eta_step={(1-n/ns)*100:5.1f}% '
                                 f'memorija={r["memorija_mb"]:8.0f}MB {r["ms_po_batchu"]:7.1f}ms')

        oslobodi('d', 'idx')
        log.info(f'  graf {ime} zavrsen')

    print(pd.read_csv(REZULTATI / 'faza1.csv').tail(12).to_string(index=False))                       

2026-07-29 23:04:21,940 [INFO] ======================================================================
2026-07-29 23:04:21,941 [INFO] FAZA 1 | graf arxiv
2026-07-29 23:04:23,259 [INFO]   169,343 cvorova, 2,315,598 potega, 128 atributa, 40 klasa, multi-label False
2026-07-29 23:04:23,263 [INFO]   ulazni stepen: prosek 13.67, medijana 6, najveci 13161
2026-07-29 23:04:23,360 [INFO]   mali S=[5, 5, 5]: granica po cvoru naivna 156, sa stepenom 117, efektivni fanout [3.93, 4.86, 4.86]
2026-07-29 23:05:04,742 [INFO]     B=   32 fp32: N=     1961 eta_naiv= 60.7% eta_step= 47.5% memorija=      51MB     5.7ms
2026-07-29 23:05:46,317 [INFO]     B=   32 fp16: N=     1961 eta_naiv= 60.7% eta_step= 47.5% memorija=      40MB     6.4ms
2026-07-29 23:06:27,548 [INFO]     B=   32 bf16: N=     1961 eta_naiv= 60.7% eta_step= 47.5% memorija=      40MB     5.5ms
2026-07-29 23:07:08,643 [INFO]     B=  128 fp32: N=     7214 eta_naiv= 63.9% eta_step= 51.7% memorija=     116MB     6.7ms
2026-07-29 23:07:49,867 

## Faza 2: sistemske optimizacije

Meri se broj radnih procesa pri učitavanju, prevođenje modela i format TF32, na dva grafa različite gustine. Merenja se ponavljaju tri puta, jer je čvor deljen pa vremena variraju.

In [7]:
F2_GRAFOVI = ['arxiv', 'products']
F2_S, F2_B = [15, 10, 5], 512
F2_PONAVLJANJA = 3
F2_PROCESI = [0, 2, 4, 8, 16, 32]
F2_UZORAKA = 100

if FAZA_2:
    uradjeno = vec_uradjeno('faza2.csv', ['graf', 'sta', 'vrednost', 'ponavljanje'])
    for ime in F2_GRAFOVI:
        log.info('=' * 70)
        log.info(f'FAZA 2 | graf {ime}, S={F2_S}, B={F2_B}')
        d, idx, klasa, multi = ucitaj_graf(ime)

        def zabelezi(sta, vrednost, p, r):
            upisi_red('faza2.csv', {
                'graf': ime, 'sta': sta, 'vrednost': str(vrednost), 'ponavljanje': p,
                'ms_po_batchu': r['ms_po_batchu'], 'memorija_mb': r['memorija_mb'],
                'cvorova': r['cvorova'], 'oom': r['oom']})
            log.info(f'  {sta}={vrednost} ponavljanje {p}: {r["ms_po_batchu"]:7.1f} ms po batch-u, '
                     f'{r["memorija_mb"]:8.0f} MB')

        for nw in F2_PROCESI:
            for p in range(F2_PONAVLJANJA):
                if (ime, 'radnih_procesa', str(nw), str(p)) in uradjeno:
                    continue
                try:
                    r = izmeri(d, idx, klasa, multi, F2_S, F2_B, 'fp32',
                               num_workers=nw, n_uzoraka=F2_UZORAKA)
                    zabelezi('radnih_procesa', nw, p, r)
                except Exception as e:
                    log.warning(f'  radnih_procesa={nw}: {type(e).__name__}: {str(e)[:90]}')
                    break

        for tf32 in [False, True]:
            for p in range(F2_PONAVLJANJA):
                if (ime, 'tf32', str(tf32), str(p)) in uradjeno:
                    continue
                r = izmeri(d, idx, klasa, multi, F2_S, F2_B, 'fp32',
                           tf32=tf32, n_uzoraka=F2_UZORAKA)
                zabelezi('tf32', tf32, p, r)

        for comp in [False, True]:
            for p in range(F2_PONAVLJANJA):
                if (ime, 'compile', str(comp), str(p)) in uradjeno:
                    continue
                try:
                    r = izmeri(d, idx, klasa, multi, F2_S, F2_B, 'fp32',
                               koristi_compile=comp, n_uzoraka=F2_UZORAKA, zagrevanje=25)
                    zabelezi('compile', comp, p, r)
                except Exception as e:
                    log.warning(f'  compile={comp}: {type(e).__name__}: {str(e)[:90]}')
                    break

        oslobodi('d', 'idx')

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print(pd.read_csv(REZULTATI / 'faza2.csv').to_string(index=False))

2026-07-30 00:12:09,542 [INFO] ======================================================================
2026-07-30 00:12:09,543 [INFO] FAZA 2 | graf arxiv, S=[15, 10, 5], B=512
2026-07-30 00:12:39,356 [INFO]   radnih_procesa=0 ponavljanje 0:   267.0 ms po batch-u,      685 MB
2026-07-30 00:13:08,207 [INFO]   radnih_procesa=0 ponavljanje 1:   262.1 ms po batch-u,      685 MB
2026-07-30 00:13:36,407 [INFO]   radnih_procesa=0 ponavljanje 2:   259.9 ms po batch-u,      685 MB
2026-07-30 00:13:52,864 [INFO]   radnih_procesa=2 ponavljanje 0:    54.6 ms po batch-u,      683 MB
2026-07-30 00:14:09,360 [INFO]   radnih_procesa=2 ponavljanje 1:    54.1 ms po batch-u,      683 MB
2026-07-30 00:14:24,796 [INFO]   radnih_procesa=2 ponavljanje 2:    42.4 ms po batch-u,      683 MB
2026-07-30 00:14:48,042 [INFO]   radnih_procesa=4 ponavljanje 0:    22.1 ms po batch-u,      687 MB
2026-07-30 00:15:11,437 [INFO]   radnih_procesa=4 ponavljanje 1:    22.8 ms po batch-u,      688 MB
2026-07-30 00:15:34,861 [

## Sažetak

In [8]:
json.dump({
    'datum': datetime.now().isoformat(),
    'kartica': torch.cuda.get_device_name(0),
    'compute_capability': f'{cc[0]}.{cc[1]}',
    'jezgara': os.cpu_count(),
    'deljena_memorija_gb': round(shutil.disk_usage('/dev/shm').total / 1e9, 1),
    'seed': SEED, 'grafovi': GRAFOVI, 'skriveni': SKRIVENI, 'lr': LR,
    'uzoraka_faza1': N_UZORAKA, 'uzoraka_faza2': F2_UZORAKA,
    'preciznosti': PRECIZNOSTI, 'batchevi': BATCHEVI,
}, open(REZULTATI / 'sazetak.json', 'w'), indent=2, default=str)

for f in ['faza1.csv', 'faza1_stepen.csv', 'faza2.csv']:
    p = REZULTATI / f
    if p.exists():
        print(f'{f}: {len(pd.read_csv(p))} redova')
print(f'\nsve u {REZULTATI}')

faza1.csv: 108 redova
faza1_stepen.csv: 6 redova
faza2.csv: 54 redova

sve u rezultati_prosirenje
